In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

In [ ]:
my_string = "the cat sat on the mat"
vocab = ["<pad>", "<unk>", "the", "cat", "sat", "on", "mat"]

T = len(my_string.split())
vocab_size = len(vocab)
d_m = 4
d_k = 4
d_v = 4
d_ff = 4 * d_m

In [ ]:
vocab = ["<pad>", "<unk>", "the", "cat", "sat", "on", "mat"]

token_to_id = {token : id for id, token in enumerate(vocab)}
id_to_token = {id : token for token, id in token_to_id.items()}

tokens = my_string.split()
# tokens = [token_to_id[token.lower()] for token in my_string.split()]
token_ids = torch.tensor([token_to_id.get(token.lower(), "<unk>") for token in tokens])

In [ ]:
def layer_norm(x: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    return x_norm

In [2]:
h = 1
d_m = 16
d_k = int(d_m / h)
d_v = int(d_m / h)

T = 32

X = torch.randn((T, d_m))

matrices = {}
qkvs = {}
attentions = []
for i in range(h):
    matrices[f"W_Q_{i}"] = torch.randn((d_m, d_k))
    matrices[f"W_K_{i}"] = torch.randn((d_m, d_k))
    matrices[f"W_V_{i}"] = torch.randn((d_m, d_v))

    Q = X @ matrices[f"W_Q_{i}"]
    K = X @ matrices[f"W_K_{i}"]
    V = X @ matrices[f"W_V_{i}"]
    qkvs[f"Q_{i}"] = Q
    qkvs[f"K_{i}"] = K
    qkvs[f"V_{i}"] = V

    mask = torch.triu(torch.ones((T, T)) * -torch.inf, diagonal=1)
    A = torch.softmax((Q @ K.T) / math.sqrt(d_k) + mask, dim=1) @ V
    attentions.append(A)

out = torch.tensor(attentions[0])
for i in range(1, len(attentions)):
    out = torch.cat((out, attentions[i]), dim=1)

W_O = torch.randn((d_m, d_m))
out = out @ W_O


/tmp/ipykernel_228566/2532078590.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  out = torch.tensor(attentions[0])


In [3]:
Q.reshape((T, h, int(d_m / h))).permute(1, 0, 2).shape

torch.Size([1, 32, 16])

In [5]:
K = Q.clone()

In [6]:
K == Q

tensor([[True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,